In [ ]:
!pip install torch torchvision torchaudio
!pip install diffusers transformers accelerate
!pip install pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 123.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 82.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 61.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 84.0 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitli

In [ ]:
import torch
import numpy as np
from PIL import Image
import base64
import io
from diffusers import StableDiffusionPipeline
#from google.colab import files
import matplotlib.pyplot as plt

In [ ]:
# Step 1: Specify a local folder containing your images
import os

image_folder = "images"  # Set this to your folder with images
image_files = [f for f in os.listdir(image_folder) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
print(f"Found {len(image_files)} images: {image_files}")

Saving WhatsApp Image 2024-04-22 at 07.26.18_1ab32dd9.jpg to WhatsApp Image 2024-04-22 at 07.26.18_1ab32dd9.jpg
Saving WhatsApp Image 2024-04-22 at 07.28.17_22e9ec8b.jpg to WhatsApp Image 2024-04-22 at 07.28.17_22e9ec8b.jpg
Saving WhatsApp Image 2024-06-12 at 20.10.22_d03908c0.jpg to WhatsApp Image 2024-06-12 at 20.10.22_d03908c0.jpg


In [3]:
# Step 2: Load Stable Diffusion Pipeline
pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5", torch_dtype=torch.float16
).to("cuda")
vae = pipe.vae.half().to("cuda")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/4.72k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/342 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.06M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/547 [00:00<?, ?B/s]

diffusion_pytorch_model.safetensors:   0%|          | 0.00/335M [00:00<?, ?B/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

In [5]:
import os
# Step 3: Create folders
os.makedirs("latents", exist_ok=True)
os.makedirs("reconstructed_images", exist_ok=True)

In [7]:
import json

# Step 4: Encode images and save latent vectors
for filename in uploaded:
    try:
        # Load and preprocess image
        image = Image.open(filename).convert("RGB").resize((512, 512))
        image_tensor = torch.tensor(np.array(image)).float() / 255.0
        image_tensor = image_tensor.permute(2, 0, 1).unsqueeze(0).to("cuda").half()

        # Encode to latent and scale
        with torch.no_grad():
            latent = vae.encode(image_tensor).latent_dist.sample()
        latent_scaled = latent * 0.18215

        # Save latent vector to JSON
        latent_np = latent_scaled.detach().cpu().numpy()
        json_data = {
            "latent_vector": latent_np.tolist(),
            "shape": latent_np.shape,
            "dtype": str(latent_np.dtype)
        }
        json_filename = os.path.splitext(filename)[0] + "_latent.json"
        with open(f"latents/{json_filename}", "w") as f:
            json.dump(json_data, f)

        print(f"✅ Latent saved: {json_filename}")

    except Exception as e:
        print(f"❌ Failed to encode {filename}: {e}")

✅ Latent saved: WhatsApp Image 2024-04-22 at 07.26.18_1ab32dd9_latent.json
✅ Latent saved: WhatsApp Image 2024-04-22 at 07.28.17_22e9ec8b_latent.json
✅ Latent saved: WhatsApp Image 2024-06-12 at 20.10.22_d03908c0_latent.json


In [ ]:
import zipfile
# Step 5: Zip all latent JSONs
zip_latents = "latent_vectors.zip"
with zipfile.ZipFile(zip_latents, "w") as zipf:
    for file in os.listdir("latents"):
        zipf.write(os.path.join("latents", file), arcname=file)

print(f"Latent vectors zipped and saved as {zip_latents}.")
# You can manually download the zip from your working directory if needed.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📦 All latent vectors zipped and downloaded.


RECONSTRUCTION


In [11]:
# Step 7: Decode latents and reconstruct images
for file in os.listdir("latents"):
    if file.endswith(".json"):
        try:
            with open(os.path.join("latents", file), "r") as f:
                latent_json = json.load(f)

            latent_vector = np.array(latent_json["latent_vector"], dtype=np.float16)
            latent_tensor = torch.tensor(latent_vector).to("cuda").half()

            latent_tensor = latent_tensor / 0.18215  # Reverse scaling
            with torch.no_grad():
                reconstructed = vae.decode(latent_tensor).sample

            reconstructed_image = reconstructed[0].detach().cpu().permute(1, 2, 0).clamp(0, 1).numpy()
            reconstructed_image_pil = Image.fromarray((reconstructed_image * 255).astype(np.uint8))

            recon_path = f"reconstructed_images/{file.replace('_latent.json', '_reconstructed.png')}"
            reconstructed_image_pil.save(recon_path)
            print(f"🖼️ Reconstructed saved: {recon_path}")
        except Exception as e:
            print(f"❌ Error reconstructing from {file}: {e}")

🖼️ Reconstructed saved: reconstructed_images/WhatsApp Image 2024-04-22 at 07.26.18_1ab32dd9_reconstructed.png
🖼️ Reconstructed saved: reconstructed_images/WhatsApp Image 2024-04-22 at 07.28.17_22e9ec8b_reconstructed.png
🖼️ Reconstructed saved: reconstructed_images/WhatsApp Image 2024-06-12 at 20.10.22_d03908c0_reconstructed.png
